# Backbone Width Ablation: 512 vs 1024\n\nIdentical grid search across both architectures. No SAM, no attention — isolating the effect of backbone width alone.\n\n**512 Backbone:** 6000->512->256->128 (~3.3M params)\n**1024 Backbone:** 6000->1024->512->256->128 (~6.5M params)\n\n**Grid:** LR [5e-5, 1e-4] x Dropout [0.3, 0.8], 6x6 = 36 combos each, 72 total.\n**Data:** Same 26,642-sample aggregated split from 04 cache.

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed")
else:
    DATA_ROOT = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet/Processed")

OUT_DIR = Path("./results_wider_ablation")
OUT_DIR.mkdir(exist_ok=True)

# Reuse cached data from 04
CACHE_PATH = Path("../04-MultiLabel-CLassifier/aggregated_multilabel_data.npz")

DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]

print(f"Drugs: {len(DRUGS_10)}  |  Cache: {CACHE_PATH}")

In [ ]:
# =============================================================================
# 1. DATASET + BASE MLP CLASS (no SAM, no attention — clean ablation)
# =============================================================================

class MultiLabelDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(np.nan_to_num(Y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.mask[idx]


class MultiLabelMaldiMLP(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_dim=512, head_dims=(256, 128),
                 dropout_high=0.3, dropout_low=0.2,
                 learning_rate=1e-3, weight_decay=0.0,
                 batch_size=64, epochs=50, early_stopping_patience=10,
                 warmup_epochs=0, val_fraction=0.1,
                 random_state=42, verbose=False, device="cpu"):
        self.hidden_dim = hidden_dim; self.head_dims = head_dims
        self.dropout_high = dropout_high; self.dropout_low = dropout_low
        self.learning_rate = learning_rate; self.weight_decay = weight_decay
        self.batch_size = batch_size; self.epochs = epochs
        self.early_stopping_patience = early_stopping_patience
        self.warmup_epochs = warmup_epochs; self.val_fraction = val_fraction
        self.random_state = random_state; self.verbose = verbose; self.device = device

    def _build_model(self, input_dim, n_labels):
        return SpectralAttentionMLP(
            input_dim=input_dim, n_classes=n_labels,
            hidden_dim=self.hidden_dim, head_dims=self.head_dims,
            use_attention=False,
            dropout_high=self.dropout_high, dropout_low=self.dropout_low)

    def fit(self, X, Y):
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        self.n_labels_ = Y.shape[1]
        self.model_ = self._build_model(X.shape[1], self.n_labels_).to(self.device)

        ds = MultiLabelDataset(X, Y)
        n_val = max(1, int(len(ds) * self.val_fraction))
        n_tr = len(ds) - n_val
        train_ds, val_ds = random_split(ds, [n_tr, n_val],
            generator=torch.Generator().manual_seed(self.random_state))
        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size * 2, shuffle=False)

        opt_cls = torch.optim.AdamW if self.weight_decay > 0 else torch.optim.Adam
        optimizer = opt_cls(self.model_.parameters(), lr=self.learning_rate,
                            weight_decay=self.weight_decay)
        warmup = max(0, self.warmup_epochs)
        t_max = max(1, self.epochs - warmup)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=t_max, eta_min=1e-6)
        criterion = nn.BCEWithLogitsLoss(reduction="none")

        best_val_loss = float("inf"); best_state = None; patience_counter = 0

        for epoch in range(self.epochs):
            self.model_.train(); train_loss = 0.0
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                if epoch < warmup:
                    for pg in optimizer.param_groups:
                        pg["lr"] = self.learning_rate * (epoch + 1) / warmup
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss = (loss * mb).sum() / mb.sum()
                loss.backward(); optimizer.step()
                train_loss += loss.item()
            if epoch >= warmup: scheduler.step()

            self.model_.eval(); val_loss = 0.0
            with torch.no_grad():
                for xb, yb, mb in val_loader:
                    xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    val_loss += loss.item()
            val_loss /= len(val_loader); train_loss /= len(train_loader)

            if self.verbose:
                print(f"  Epoch {epoch+1:3d}/{self.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    if self.verbose: print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state is not None: self.model_.load_state_dict(best_state)
        self.model_.eval(); self.is_fitted_ = True
        return self

    def predict_proba(self, X):
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            return torch.sigmoid(self.model_(X_t)).cpu().numpy()

    def predict(self, X, thresholds=None):
        proba = self.predict_proba(X)
        if thresholds is None: thresholds = [0.5] * self.n_labels_
        return (proba >= np.array(thresholds)).astype(int)

---\n## Load Data + Split

In [ ]:
# =============================================================================
# 2. LOAD DATA + SPLIT (70/15/15)
# =============================================================================

if CACHE_PATH.exists():
    print(f"Loading from cache: {CACHE_PATH}")
    cached = np.load(CACHE_PATH, allow_pickle=True)
    X_all = cached['X_all']; Y_all = cached['Y_all']; sp_all = cached['sp_all']
else:
    raise FileNotFoundError(f"Cache not found: {CACHE_PATH}. Run 04-MultiLabel first.")

print(f"Total: {X_all.shape[0]} samples, {len(np.unique(sp_all))} species")

idx_trval, idx_test, _, _ = stratified_species_drug_split(
    np.arange(len(Y_all)).reshape(-1, 1), np.zeros(len(Y_all)),
    species=sp_all, test_size=0.15, random_state=SEED)
idx_trval = idx_trval.flatten().astype(int); idx_test = idx_test.flatten().astype(int)
X_trval, Y_trval, sp_trval = X_all[idx_trval], Y_all[idx_trval], sp_all[idx_trval]
X_test, Y_test = X_all[idx_test], Y_all[idx_test]

val_frac = 0.15 / 0.85
idx_train, idx_val, _, _ = stratified_species_drug_split(
    np.arange(len(Y_trval)).reshape(-1, 1), np.zeros(len(Y_trval)),
    species=sp_trval, test_size=val_frac, random_state=SEED)
idx_train = idx_train.flatten().astype(int); idx_val = idx_val.flatten().astype(int)
X_train, Y_train = X_trval[idx_train], Y_trval[idx_train]
X_val, Y_val = X_trval[idx_val], Y_trval[idx_val]
print(f"Split: train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

state = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state); X_val_pp = apply_input_transform(X_val, state)
X_test_pp = apply_input_transform(X_test, state)
print("Preprocessing done.")

---\n## Grid Search: 512 Backbone

In [ ]:
# =============================================================================
# 3. GRID SEARCH: 512 BACKBONE
# =============================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LR_GRID   = np.linspace(5e-5, 1e-4, 6)
DROP_GRID = np.linspace(0.3, 0.8, 6)
thresholds = np.linspace(0.05, 0.95, 91)
print(f"Grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")
print("Backbone: 512 -> 256 -> 128")

best_512_balacc = -1; best_512_lr = None; best_512_dh = None; best_512_dl = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=512, head_dims=(256, 128),
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))
        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        mark = " *" if macro_ba > best_512_balacc else ""
        print(f"  512 lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > best_512_balacc:
            best_512_balacc = macro_ba; best_512_lr = lr; best_512_dh = dh; best_512_dl = dl

print(f"\nBest 512: lr={best_512_lr:.1e} drop=({best_512_dh:.2f},{best_512_dl:.2f})  macro_BalAcc={best_512_balacc:.4f}")

---\n## Retrain Best 512

In [ ]:
# =============================================================================
# 4. RETRAIN BEST 512
# =============================================================================

mlp_512 = MultiLabelMaldiMLP(
    hidden_dim=512, head_dims=(256, 128),
    dropout_high=best_512_dh, dropout_low=best_512_dl,
    learning_rate=best_512_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_512.fit(X_train_pp, Y_train)

proba_val_512 = mlp_512.predict_proba(X_val_pp)
thresh_512 = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2: thresh_512.append(0.5); continue
    pv = proba_val_512[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    thresh_512.append(bt)
thresh_512 = np.array(thresh_512)

proba_test_512 = mlp_512.predict_proba(X_test_pp)
results_512 = {}
for di, drug in enumerate(DRUGS_10):
    tm = ~np.isnan(Y_test[:, di])
    if tm.sum() < 2: results_512[drug] = np.nan; continue
    preds = (proba_test_512[tm, di] >= thresh_512[di])
    yt = Y_test[tm, di].astype(int)
    results_512[drug] = balanced_accuracy_score(yt, preds)

print("\n512 Backbone -- Test BalAcc:")
for drug, ba in results_512.items():
    print(f"  {drug:35s}  {ba:.4f}")
print(f"  {'MACRO AVG':35s}  {np.nanmean(list(results_512.values())):.4f}")

---\n## Grid Search: 1024 Backbone

In [ ]:
# =============================================================================
# 5. GRID SEARCH: 1024 BACKBONE
# =============================================================================

print("Backbone: 1024 -> 512 -> 256 -> 128")

best_1024_balacc = -1; best_1024_lr = None; best_1024_dh = None; best_1024_dl = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=1024, head_dims=(512, 256, 128),
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))
        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        mark = " *" if macro_ba > best_1024_balacc else ""
        print(f"  1024 lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > best_1024_balacc:
            best_1024_balacc = macro_ba; best_1024_lr = lr; best_1024_dh = dh; best_1024_dl = dl

print(f"\nBest 1024: lr={best_1024_lr:.1e} drop=({best_1024_dh:.2f},{best_1024_dl:.2f})  macro_BalAcc={best_1024_balacc:.4f}")

---\n## Retrain Best 1024

In [ ]:
# =============================================================================
# 6. RETRAIN BEST 1024
# =============================================================================

mlp_1024 = MultiLabelMaldiMLP(
    hidden_dim=1024, head_dims=(512, 256, 128),
    dropout_high=best_1024_dh, dropout_low=best_1024_dl,
    learning_rate=best_1024_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_1024.fit(X_train_pp, Y_train)

proba_val_1024 = mlp_1024.predict_proba(X_val_pp)
thresh_1024 = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2: thresh_1024.append(0.5); continue
    pv = proba_val_1024[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    thresh_1024.append(bt)
thresh_1024 = np.array(thresh_1024)

proba_test_1024 = mlp_1024.predict_proba(X_test_pp)
results_1024 = {}
for di, drug in enumerate(DRUGS_10):
    tm = ~np.isnan(Y_test[:, di])
    if tm.sum() < 2: results_1024[drug] = np.nan; continue
    preds = (proba_test_1024[tm, di] >= thresh_1024[di])
    yt = Y_test[tm, di].astype(int)
    results_1024[drug] = balanced_accuracy_score(yt, preds)

print("\n1024 Backbone -- Test BalAcc:")
for drug, ba in results_1024.items():
    print(f"  {drug:35s}  {ba:.4f}")
print(f"  {'MACRO AVG':35s}  {np.nanmean(list(results_1024.values())):.4f}")

---\n## Comparison: 512 vs 1024

In [ ]:
# =============================================================================
# 7. COMPARISON: 512 vs 1024
# =============================================================================

comp_rows = []
for drug in DRUGS_10:
    b512 = results_512.get(drug, np.nan)
    b1024 = results_1024.get(drug, np.nan)
    delta = b1024 - b512 if not (np.isnan(b512) or np.isnan(b1024)) else np.nan
    comp_rows.append({
        "Drug": drug[:15],
        "512_Backbone": b512,
        "1024_Backbone": b1024,
        "Delta": delta,
    })
df_comp = pd.DataFrame(comp_rows)

print("\n512 vs 1024 -- Per-Drug Test BalAcc:")
print(df_comp.to_string(index=False, float_format="%.4f"))

macro_512 = np.nanmean(list(results_512.values()))
macro_1024 = np.nanmean(list(results_1024.values()))
print(f"\n  MACRO AVG  512={macro_512:.4f}  1024={macro_1024:.4f}  Delta={macro_1024-macro_512:+.4f}")
print(f"  Best LR  512={best_512_lr:.1e} drop=({best_512_dh:.2f},{best_512_dl:.2f})")
print(f"  Best LR 1024={best_1024_lr:.1e} drop=({best_1024_dh:.2f},{best_1024_dl:.2f})")

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp)); w = 0.35
ax.bar(x - w/2, df_comp["512_Backbone"], w, label="512 Backbone", color="#aec7e8")
ax.bar(x + w/2, df_comp["1024_Backbone"], w, label="1024 Backbone", color="#1f77b4")
ax.set_xticks(x); ax.set_xticklabels(df_comp["Drug"], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Test BalAcc"); ax.set_title("Backbone Width Ablation: 512 vs 1024")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
for i, (b512, b1024) in enumerate(zip(df_comp["512_Backbone"], df_comp["1024_Backbone"])):
    if not np.isnan(b512) and not np.isnan(b1024):
        sign = "+" if b1024 > b512 else ""
        ax.annotate(f"{sign}{b1024-b512:+.3f}", (i, max(b512, b1024) + 0.01),
                    fontsize=7, ha="center", color="red" if b1024 > b512 else "gray")
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "ablation_512_vs_1024.pdf")
plt.show()

# Also plot macro avg
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["512", "1024"], [macro_512, macro_1024], color=["#aec7e8", "#1f77b4"], width=0.5)
ax.set_ylabel("Macro Avg Test BalAcc"); ax.set_title("Backbone Width: Macro Average")
ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
for i, v in enumerate([macro_512, macro_1024]):
    ax.text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "ablation_macro_avg.pdf")
plt.show()

In [ ]:
print("\nDone. Results saved to", OUT_DIR.resolve())
import joblib
joblib.dump({
    "results_512": results_512, "results_1024": results_1024,
    "best_512": {"lr": best_512_lr, "dh": best_512_dh, "dl": best_512_dl},
    "best_1024": {"lr": best_1024_lr, "dh": best_1024_dh, "dl": best_1024_dl},
}, OUT_DIR / "ablation_results.joblib")
print("Ablation results saved.")